In [ ]:
import sys
if "google.colab" in sys.modules:
    !pip install nasap-net


In [ ]:
# Logging configuration
import logging
import sys

logger = logging.getLogger()
logger.setLevel(logging.DEBUG)


In [ ]:
# Assembly enumeration
# Enumerate substructures of M10L11

from nasap_net import Assembly, Bond, Component, enumerate_assemblies, enumerate_reactions, MLEKind

M = Component(kind='M', sites=[0, 1])
L = Component(kind='L', sites=[0, 1])
X = Component(kind='X', sites=[0])

M10L11 = Assembly(
    components={
        'M0': M, 'M1': M, 'M2': M, 'M3': M, 'M4': M,
        'M5': M, 'M6': M, 'M7': M, 'M8': M, 'M9': M,
        'L0': L, 'L1': L, 'L2': L, 'L3': L, 'L4': L,
        'L5': L, 'L6': L, 'L7': L, 'L8': L, 'L9': L, 'L10': L
    },
    bonds=[
        Bond('L0', 1, 'M0', 0), Bond('M0', 1, 'L1', 0),
        Bond('L1', 1, 'M1', 0), Bond('M1', 1, 'L2', 0),
        Bond('L2', 1, 'M2', 0), Bond('M2', 1, 'L3', 0),
        Bond('L3', 1, 'M3', 0), Bond('M3', 1, 'L4', 0),
        Bond('L4', 1, 'M4', 0), Bond('M4', 1, 'L5', 0),
        Bond('L5', 1, 'M5', 0), Bond('M5', 1, 'L6', 0),
        Bond('L6', 1, 'M6', 0), Bond('M6', 1, 'L7', 0),
        Bond('L7', 1, 'M7', 0), Bond('M7', 1, 'L8', 0),
        Bond('L8', 1, 'M8', 0), Bond('M8', 1, 'L9', 0),
        Bond('L9', 1, 'M9', 0), Bond('M9', 1, 'L10', 0),
    ]
)

assemblies = list(enumerate_assemblies(
    template=M10L11,
    leaving_ligand=X,
    metal_kinds='M',
))

logger.info('Assembly enumeration completed!')
logger.info('Assembly count: %i', len(assemblies))


In [ ]:
# Add assemblies not present as M10L11 substructures but needed (M3L3, M4L4)

M3L3 = Assembly(
    components={
        'M0': M, 'M1': M, 'M2': M,
        'L0': L, 'L1': L, 'L2': L,
    },
    bonds=[
        Bond('M0', 1, 'L0', 0), Bond('L0', 1, 'M1', 0),
        Bond('M1', 1, 'L1', 0), Bond('L1', 1, 'M2', 0),
        Bond('M2', 1, 'L2', 0), Bond('L2', 1, 'M0', 0),
    ]
)

M4L4 = Assembly(
    components={
        'M0': M, 'M1': M, 'M2': M, 'M3': M,
        'L0': L, 'L1': L, 'L2': L, 'L3': L,
    },
    bonds=[
        Bond('M0', 1, 'L0', 0), Bond('L0', 1, 'M1', 0),
        Bond('M1', 1, 'L1', 0), Bond('L1', 1, 'M2', 0),
        Bond('M2', 1, 'L2', 0), Bond('L2', 1, 'M3', 0),
        Bond('M3', 1, 'L3', 0), Bond('L3', 1, 'M0', 0),
    ]
)

assemblies.extend([M3L3, M4L4])


In [ ]:
# Assign IDs to assemblies based on composition formula
# Assemblies sharing the same formula are numbered sequentially.

from nasap_net.helpers import assign_composition_formula_ids

assemblies = assign_composition_formula_ids(assemblies, order=['M', 'L', 'X'])


In [ ]:
# Elementary reaction enumeration

import time

start = time.perf_counter()

reaction_iter = enumerate_reactions(
    assemblies=assemblies,
    mle_kinds=[
        MLEKind(metal='M', leaving='X', entering='L'),
        MLEKind(metal='M', leaving='L', entering='X'),
        MLEKind(metal='M', leaving='L', entering='L'),
        MLEKind(metal='M', leaving='X', entering='X'),
    ],
    min_temp_ring_size=3
)
reactions = list(reaction_iter)

end = time.perf_counter()

logger.info('Reaction enumeration completed!')
logger.info('Reaction count: %i', len(reactions))
logger.info('Elapsed time: %.2f s', end - start)


In [ ]:
# Exclude reactions that transiently form 5- to 10-membered rings
# This filtering is specific to this system; skip this cell for other systems.

reactions = [
    reaction
    for reaction in reactions
    if not (
        reaction.as_reaction_to_classify().forming_ring_size_including_temporary
        in {5, 6, 7, 8, 9, 10}
    )
]

print(len(reactions))


In [ ]:
# Save assemblies and reactions to files

import os
from nasap_net import save_assemblies, save_reactions

os.makedirs("output", exist_ok=True)

ASSEMBLY_PATH = "output/assemblies.yaml"
REACTION_PATH = "output/reactions.csv"

save_assemblies(assemblies, ASSEMBLY_PATH, overwrite=True)
save_reactions(reactions, REACTION_PATH, overwrite=True)


In [ ]:
# Load assemblies and reactions from files
# Start from this cell if you already have the files and want to skip to classification.
# If you have not run the enumeration yet, copy the files from expected/ to output/ first.

from nasap_net import load_assemblies, load_reactions

ASSEMBLY_PATH = "output/assemblies.yaml"
REACTION_PATH = "output/reactions.csv"

assemblies = load_assemblies(ASSEMBLY_PATH)
reactions = load_reactions(
    REACTION_PATH,
    assemblies=assemblies,
    site_id_type="int"
)


In [ ]:
# Define the reaction classification rule
# Start from this cell if you already have the files and want to classify reactions only.

# This cell only defines the classifier function; run the next cell to apply it.

from nasap_net import Assembly, Reaction, IncompleteReactionClassifierError


class InvalidReactionError(ValueError):
    pass


# Classifier function
def classify_reaction(reaction: Reaction) -> str:
    # Always call this at the start of the classifier
    reaction = reaction.as_reaction_to_classify()

    # X-L exchange
    if reaction.leaving_kind == 'X' and reaction.entering_kind == 'L':

        # Classify by ring-formation size
        if reaction.forms_ring():
            if reaction.forming_ring_size == 4:
                return '6f'
            elif reaction.forming_ring_size == 3:
                return '8f'
            else:
                pass

        # Classify by ligand/metal counts before reaction
        # Note: site index and count differ by 1
        # e.g., "L enters M's 2nd site" means init_ligand_count_on_metal == 1

        init_L_count_on_M = reaction.init_ligand_count_on_metal
        init_M_count_on_L = reaction.init_metal_count_on_ligand

        if init_L_count_on_M == 0 and init_M_count_on_L == 0:
            return '1f'
        if init_L_count_on_M == 0 and init_M_count_on_L == 1:
            return '2f'
        if init_L_count_on_M == 1 and init_M_count_on_L == 0:
            return '3f'
        if init_L_count_on_M == 1 and init_M_count_on_L == 1:
            return '4f'

        # Raise IncompleteReactionClassifierError for cases you believe are unreachable.
        # If execution reaches here, it signals an incomplete classification rule.
        raise IncompleteReactionClassifierError(reaction, 'Unknown X-L')

    # L-X exchange
    elif reaction.leaving_kind == 'L' and reaction.entering_kind == 'X':
        # Get the reverse reaction
        rev = reaction.sample_rev
        # Classify the reverse reaction
        rev_type = classify_reaction(rev)

        # Map reverse type to forward type
        if rev_type == '1f':
            return '1b'
        if rev_type == '2f':
            return '2b'
        if rev_type == '3f':
            return '3b'
        if rev_type == '4f':
            return '4b'
        if rev_type == '6f':
            return '6b'
        if rev_type == '8f':
            return '8b'

        raise IncompleteReactionClassifierError(reaction, 'Unknown L-X')

    # L-L exchange
    elif reaction.leaving_kind == 'L' and reaction.entering_kind == 'L':
        # Classify by ring-formation size
        if reaction.forms_ring():
            if reaction.forming_ring_size == 4:
                return '7f'
            elif reaction.forming_ring_size == 3:
                return '9f'
            else:
                pass

        # Classify by ring-breaking size
        if reaction.breaks_ring():
            if reaction.breaking_ring_size == 4:
                return '7b'
            elif reaction.breaking_ring_size == 3:
                return '9b'
            else:
                pass

        return '5'

    # X-X exchange
    elif reaction.leaving_kind == 'X' and reaction.entering_kind == 'X':
        return '10'

    raise IncompleteReactionClassifierError(reaction, 'Unknown')



In [ ]:
# Run reaction classification

from nasap_net import classify_reactions

reaction_to_class = classify_reactions(reactions, classify_reaction)

logger.info('Classification completed!')


In [ ]:
# Summarize classification results (optional)
# Count how many reactions were assigned to each class.

from collections import Counter

counter = Counter(reaction_to_class.values())
for reaction_class, count in sorted(counter.items()):
    print(f'{reaction_class}: {count}')


In [ ]:
# Save classification results to a file

import os
from nasap_net import save_classification_result

os.makedirs("output", exist_ok=True)

CLASSIFICATION_RESULT_PATH = "output/classification_result.csv"

save_classification_result(reaction_to_class, CLASSIFICATION_RESULT_PATH, overwrite=True)
